# MOLTIE – Jupyter Inference Cell (y_spec + Needle Runner)

## Purpose

This notebook cell executes the `y_spec.py` schema inference module **and** a closed-set needle classifier over each row of a witness statement CSV file. It is designed for **interactive debugging, enrichment, and calibration**, not bulk production execution.

It enables row-level inspection of semantic tagging (needles), structured Y inference, JSON validity, and stability before transitioning to a production batch workflow.

---

## Input

* **CSV file**:
  `/home/hello/Projects/Statements/input/Leonardo_WS.csv`

* **Column used**:
  `text_verbatim`

Each row from `text_verbatim` is treated as `ws_text` and:

1. Passed via `stdin` to `y_spec.py`
2. Independently evaluated by the needle classifier (LLM closed-set tagging)

---

## Outputs

Two files are written to:

`/home/hello/Projects/Statements/output`

1. **Leonardo_WS_enhanced.csv**

   * Contains **all original CSV columns**
   * Adds:

     * `needle_selected_raw`
     * `has__/conf__/quote__` columns per tag
     * `y_ok`, `y_rc`
     * `ws_len`
     * `X1` (1-based row id)
     * `doc`

2. **Y_inferred.json**

   * Aggregates structured `y_spec` output per processed row
   * Preserves full Y JSON for auditability

---

## Execution Flow

### 1. Configuration

The cell defines:

* Model name (`mistral-small3.2:latest`)
* Path to `y_spec.py`
* Debug mode toggle
* Flexible row selector (`DEBUG_SLICE`)
* Needle tag taxonomy (closed set)

Debug selector supports:

* `"3"` → process exactly the 3rd row (1-based)
* `":5"` → first 5 rows
* `"2:5"` → Python slice semantics

This allows precise surgical debugging.

---

### 2. CSV Loading

* Reads the CSV using pandas
* Validates `text_verbatim` exists
* Converts nulls to empty strings
* Determines which rows to process (full run or debug slice)

Original CSV structure is preserved.

---

### 3. Row Iteration (with tqdm)

For each selected row:

* Assigns a deterministic 1-based identifier (`X1`)
* Strips whitespace
* Skips empty rows
* Prints trace:

```
X1=<row_number> doc=Leonardo_WS.csv
```

This guarantees reproducibility and traceability.

---

### 4. Dual Processing Per Row

Each `ws_text` flows through two independent pipes:

#### A) Needle Classifier (Semantic Tagging)

* Closed-set LLM classification
* Evidence-quoted
* Negation-aware
* JSON-validated
* Returns:

  * Selected tags
  * Confidence scores
  * Evidence quotes

Expanded into structured columns (`has__/conf__/quote__`).

#### B) Y-Spec Structured Inference

```
python y_spec.py --model mistral-small3.2:latest
```

* Input via `stdin`
* Captures:

  * `stdout`
  * `stderr`
  * `returncode`
* Attempts immediate JSON parsing
* Aggregates valid results into `Y_inferred.json`

Needle tagging and Y inference remain structurally independent.

---

### 5. Enrichment Merge

The enrichment fields are merged back into the **full original DataFrame**, ensuring:

* No original data is lost
* Only processed rows receive populated enrichment fields (in debug mode)
* Non-processed rows remain intact

---

## Why This Design Is Intentional

This notebook version prioritises:

* Structural independence between retrieval (needles) and reasoning (Y)
* Full preservation of source data
* Evidence-anchored tagging
* Deterministic debug slicing
* Immediate JSON validation
* Transparent failure surfaces

It does **not** parallelise.

It does **not** bias X/Y extraction toward needle concepts.

It maintains clean architectural separation between semantic tagging and structured inference.

---

## When to Transition to Production Script

Move to a full `.py` batch runner once:

* Needle classification is stable
* Y JSON schema is consistent
* Debug slicing no longer required
* Error patterns are understood

Production version should then:

* Support parallel execution
* Implement retry logic
* Log failures explicitly
* Optionally write per-row Y JSON files

---

## Role in the MOLTIE Architecture

This cell functions as a **calibration and enrichment harness**.

It sits between:

* Raw witness statement substrate
* Independent semantic tagging layer
* Structured Y inference layer

It ensures that:

* Retrieval signals (needles)
* Structural reasoning signals (Y)

are derived independently from the same text.

---

## Summary

This Jupyter cell provides a controlled inference environment that:

* Reads witness statement rows
* Applies independent needle classification
* Executes `y_spec.py`
* Validates structured JSON output
* Merges enrichment into the original dataset
* Writes consolidated outputs

It is modular, auditable, deterministic under debug, and structurally clean.


In [2]:
import json
import subprocess
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

# =========================================================
# CONFIG
# =========================================================
CSV_PATH = Path("/home/hello/Projects/Statements/input/Leonardo_WS.csv")
TEXT_COL = "text_verbatim"

OUT_DIR = Path("/home/hello/Projects/Statements/output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ENHANCED_CSV = OUT_DIR / "Leonardo_WS_enhanced.csv"
OUT_Y_JSON = OUT_DIR / "Y_inferred.json"

Y_SPEC_PY = Path("/home/hello/Projects/Statements/code/moltie/schemas/y_spec.py")
MODEL = "mistral-small3.2:latest"
OLLAMA_CMD = ["ollama", "run", MODEL]

DEBUG = True
DEBUG_SLICE = "3"   # examples: "3" (3rd row only), ":5" (first 5), "2:5" etc.

# Needle tags (closed set)
ALLOWED_TAGS = [
    "upheld",
    "verbal_warning",
    "no_contemporaneous_evidence",
    "predetermination",
    "appeal_scope_limitation",
    "none",
]
TAG_DEFS = {
    "upheld": "appeal upheld / allowed / succeeds",
    "verbal_warning": "verbal warning mentioned as a disciplinary step or fact",
    "no_contemporaneous_evidence": "absence of notes/records or finding of no contemporaneous evidence",
    "predetermination": "decision appeared predetermined / outcome fixed / mind closed",
    "appeal_scope_limitation": "point not in grounds / outside scope / refused/declined to consider / limited to grounds",
    "none": "none of the above apply",
}

# =========================================================
# HELPERS
# =========================================================
def _extract_json_object(raw: str) -> str:
    raw = (raw or "").strip()
    start = raw.find("{")
    end = raw.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object detected in output.")
    return raw[start : end + 1]

def parse_debug_selector(selector: str, n_rows: int) -> list[int]:
    """
    Supports:
      - "3" => exact 3rd row (1-based) -> index [2]
      - ":5", "2:5", "2:", ":10:2" => Python slice syntax (0-based)
    Returns list of 0-based indices to process.
    """
    s = (selector or "").strip()
    if not s:
        return list(range(n_rows))

    if s.isdigit():
        k = int(s)
        if k < 1 or k > n_rows:
            raise ValueError(f"DEBUG_SLICE='{s}' out of range 1..{n_rows}")
        return [k - 1]

    if ":" in s:
        parts = s.split(":")
        if len(parts) > 3:
            raise ValueError(f"Invalid slice syntax: '{s}'")

        def to_int(x):
            x = x.strip()
            return None if x == "" else int(x)

        start = to_int(parts[0]) if len(parts) >= 1 else None
        stop  = to_int(parts[1]) if len(parts) >= 2 else None
        step  = to_int(parts[2]) if len(parts) == 3 else None

        sl = slice(start, stop, step)
        return list(range(n_rows))[sl]

    raise ValueError(f"Unrecognized DEBUG_SLICE format: '{selector}'")

def run_y_spec(ws_text: str) -> tuple[dict | None, str, str, int]:
    """
    Runs y_spec.py reading ws_text via stdin.
    Returns: (parsed_json_or_none, stdout, stderr, returncode)
    """
    proc = subprocess.run(
        ["python", str(Y_SPEC_PY), "--model", MODEL],
        input=ws_text,
        text=True,
        capture_output=True,
    )
    stdout = (proc.stdout or "").strip()
    stderr = (proc.stderr or "").strip()

    if proc.returncode != 0:
        return None, stdout, stderr, proc.returncode

    try:
        parsed = json.loads(_extract_json_object(stdout))
        return parsed, stdout, stderr, 0
    except Exception:
        return None, stdout, stderr, 0

def make_needle_prompt(text: str) -> str:
    tag_lines = "\n".join([f'- "{t}": {TAG_DEFS[t]}' for t in ALLOWED_TAGS])
    return f"""
TASK:
Given TEXT, select all applicable TAGS from the allowed list.
Choose ONLY from the allowed tags. If none apply, select ONLY ["none"].

ALLOWED_TAGS:
{tag_lines}

RULES:
- For every selected tag (except "none"), provide:
  - confidence in [0,1]
  - evidence_quote copied verbatim from TEXT (max 200 chars)
  - negated: true if TEXT explicitly indicates the opposite
- If you cannot quote evidence from TEXT, do NOT select that tag.
- Return VALID JSON ONLY. No markdown, no commentary, no extra keys.

TEXT:
<<<
{text.strip()}
>>>

OUTPUT JSON SCHEMA:
{{
  "selected": [
    {{"tag": "...", "confidence": 0.0, "negated": false, "evidence_quote": "..."}}
  ]
}}
""".strip()

def run_needle_tagger(text: str) -> tuple[list[dict], str, str, int]:
    """
    Returns: (selected_list, stdout, stderr, returncode)
    """
    proc = subprocess.run(
        OLLAMA_CMD,
        input=make_needle_prompt(text),
        text=True,
        capture_output=True,
    )
    stdout = (proc.stdout or "").strip()
    stderr = (proc.stderr or "").strip()

    if proc.returncode != 0:
        return [], stdout, stderr, proc.returncode

    try:
        out = json.loads(_extract_json_object(stdout))
        selected = out.get("selected", [])
        if not isinstance(selected, list):
            selected = []

        tags = [d.get("tag") for d in selected if d.get("tag")]
        for t in tags:
            if t not in ALLOWED_TAGS:
                raise ValueError(f"Invalid tag returned: {t}")
        if "none" in tags and len(tags) > 1:
            raise ValueError(f'"none" must be the only tag if present. Got: {tags}')

        return selected, stdout, stderr, 0
    except Exception:
        return [], stdout, stderr, 0

def flatten_selected(selected: list[dict]) -> dict:
    """
    Expand selected tags into has__/conf__/quote__ columns.
    Ignores negated=true items by design.
    """
    rec = {}
    for t in ALLOWED_TAGS:
        if t == "none":
            continue
        rec[f"has__{t}"] = False
        rec[f"conf__{t}"] = 0.0
        rec[f"quote__{t}"] = ""

    for item in selected:
        tag = item.get("tag")
        if not tag or tag == "none":
            continue
        if item.get("negated") is True:
            continue

        rec[f"has__{tag}"] = True
        rec[f"conf__{tag}"] = float(item.get("confidence") or 0.0)
        rec[f"quote__{tag}"] = (item.get("evidence_quote") or "")[:200]

    return rec

# =========================================================
# LOAD
# =========================================================
df = pd.read_csv(CSV_PATH)

if TEXT_COL not in df.columns:
    raise KeyError(f"Column '{TEXT_COL}' not found. Found: {list(df.columns)}")

texts_all = df[TEXT_COL].fillna("").astype(str).tolist()
n_total = len(texts_all)

idxs = parse_debug_selector(DEBUG_SLICE, n_total) if DEBUG else list(range(n_total))

print(f"Total rows in CSV: {n_total}")
print(f"Processing indices (0-based): {idxs[:30]}{' ...' if len(idxs) > 30 else ''}")
print(f"Count: {len(idxs)} | DEBUG={DEBUG} | DEBUG_SLICE='{DEBUG_SLICE}'")

# =========================================================
# RUN PIPELINE (collect enrichments keyed by original row index)
# =========================================================
enrich_by_idx = {}
y_results = {
    "version": "Y_inferred_v2",
    "source": {"csv": str(CSV_PATH), "text_col": TEXT_COL, "model": MODEL},
    "rows": {}
}

for idx in tqdm(idxs, desc="Rows (Needles + y_spec)"):
    ws_text = texts_all[idx].strip()
    X1 = idx + 1  # 1-based row number

    if not ws_text:
        # still record empties for completeness
        empty_selected = [{"tag": "none", "confidence": 1.0, "negated": False, "evidence_quote": ""}]
        rec = {
            "ws_len": 0,
            "needle_selected_raw": json.dumps(empty_selected, ensure_ascii=False),
            "needle_rc": 0,
            "y_ok": False,
            "y_rc": 0,
        }
        rec.update(flatten_selected(empty_selected))
        enrich_by_idx[idx] = rec
        continue

    print(f"X1={X1} doc={CSV_PATH.name}")

    # Needles on raw text
    selected, n_stdout, n_stderr, n_rc = run_needle_tagger(ws_text)
    needle_flat = flatten_selected(selected)

    # y_spec on raw text
    y_json, y_stdout, y_stderr, y_rc = run_y_spec(ws_text)

    y_results["rows"][f"X1_{X1:04d}"] = {
        "row_index_1based": X1,
        "doc": CSV_PATH.name,
        "y_ok": bool(y_json),
        "y": y_json if y_json else None,
        "y_returncode": y_rc,
        "y_stderr_head": (y_stderr[:400] if DEBUG and y_stderr else ""),
        "y_stdout_head": (y_stdout[:400] if DEBUG and y_stdout else ""),
    }

    rec = {
        "ws_len": len(ws_text),
        "needle_selected_raw": json.dumps(selected, ensure_ascii=False),
        "needle_rc": n_rc,
        "y_ok": bool(y_json),
        "y_rc": y_rc,
    }
    rec.update(needle_flat)

    enrich_by_idx[idx] = rec

# =========================================================
# MERGE ENRICHMENT BACK INTO FULL ORIGINAL DF
#   - For rows not processed (debug mode), enrichment fields will be blank/NA
# =========================================================
# Build an enrichment DF with same index as original df (0..n-1)
enrich_df = pd.DataFrame.from_dict(enrich_by_idx, orient="index")

# Ensure index aligns to original df index
enrich_df.index.name = "row_index_0based"

df_out = df.copy()
df_out = df_out.join(enrich_df, how="left")

# Add X1 column (1-based row number) if you want it in the enhanced CSV
df_out.insert(0, "X1", df_out.index + 1)
df_out.insert(1, "doc", CSV_PATH.name)

# =========================================================
# WRITE OUTPUTS
# =========================================================
df_out.to_csv(OUT_ENHANCED_CSV, index=False)
OUT_Y_JSON.write_text(json.dumps(y_results, indent=2, ensure_ascii=False), encoding="utf-8")

print("\nWrote outputs:")
print(f" - Enhanced CSV (ALL original cols + needle cols): {OUT_ENHANCED_CSV}")
print(f" - Y_inferred.json: {OUT_Y_JSON}")

display(df_out.head(25))

Total rows in CSV: 12
Processing indices (0-based): [2]
Count: 1 | DEBUG=True | DEBUG_SLICE='3'


Rows (Needles + y_spec):   0%|          | 0/1 [00:00<?, ?it/s]

X1=3 doc=Leonardo_WS.csv

Wrote outputs:
 - Enhanced CSV (ALL original cols + needle cols): /home/hello/Projects/Statements/output/Leonardo_WS_enhanced.csv
 - Y_inferred.json: /home/hello/Projects/Statements/output/Y_inferred.json


,X1,doc,source_row,section_title,section_summary,text_verbatim,claims,evidence_mentioned,evidence_to_request,evidence_to_locate_own,...,quote__verbal_warning,has__no_contemporaneous_evidence,conf__no_contemporaneous_evidence,quote__no_contemporaneous_evidence,has__predetermination,conf__predetermination,quote__predetermination,has__appeal_scope_limitation,conf__appeal_scope_limitation,quote__appeal_scope_limitation
0,1,Leonardo_WS.csv,1,"Performance Assessment, Role Evolution, and Ma...","Between 2019 and 2020, the Claimant’s role was...","Background, Role Evolution, Performance Contex...","[\n {\n ""text"": ""Throughout the Claimant’s...","[\n ""foundational training"",\n ""queue re...",[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Leonardo_WS.csv,2,"Definition of Projects, Management Instruction...",Management expressly defined and endorsed the ...,"Definition of “Projects”, Management Instructi...",['Management raised system behaviour in July 2...,• DSAR internal management communications (Jan...,• Any internal policy or guidance warning agai...,• Claimant’s calendar or work logs evidencing ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Leonardo_WS.csv,3,WorkCentre Status Changes,Author discussed WorkCentre status changes wit...,Escalation Only After Role Change (February 20...,[],[],[],[],...,,True,0.9,the absence of any contemporaneous instruction...,False,0.0,,True,0.8,It failed to account for the materially differ...
3,4,Leonardo_WS.csv,4,WorkCentre Status Changes,Author discussed WorkCentre status changes wit...,19–20 March 2024: First Presentation of Allega...,[{'text': 'The meeting on 19 March 2024 was no...,"[""calendar invitation titled 'meeting'"", 'meet...",[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Leonardo_WS.csv,5,WorkCentre Status Changes,Author discussed WorkCentre status changes wit...,"Reassurance, Reliance, and Subsequent Conduct\...",['Ms Oteri provided reassurance on 20 March 20...,"['Statement by Ms Oteri on 20 March 2024', 'In...",[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6,Leonardo_WS.csv,6,Analysis Omissions,Respondent’s analysis fails to distinguish bet...,"Failure to Distinguish Queue Time, Absence of ...",[{'text': 'A critical omission in the Responde...,"['crash figures', 'disciplinary hearing', 'app...","['underlying system logs', 'detailed logs show...",[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,Leonardo_WS.csv,7,Disciplinary Hearing Issues,The author faced procedural unfairness due to ...,Notice of Disciplinary Hearing and Procedural ...,[{'text': 'The author received very short noti...,['chat message from Ms Parisi on 2 April 2024'...,"['ET3 document', 'crash data', 'contemporaneou...",['chat message from Ms Parisi on 2 April 2024'...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,8,Leonardo_WS.csv,8,Disciplinary Hearing Issues,The author faced procedural unfairness due to ...,"The Disciplinary Hearing: Hostile Conduct, Lin...",[{'text': 'Mr O’Hagan interrupted and prevente...,"['Written defence submitted the previous day',...","[""Documentary evidence or analysis supporting ...",['Any records or notes from the speaker regard...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9,Leonardo_WS.csv,9,Disciplinary Hearing Issues,The author faced procedural unfairness due to ...,Failure to Comply with the Respondent’s Own Di...,['The Respondent’s disciplinary policy was not...,"['Respondent’s disciplinary policy', 'Chat mes...",['Details of the Respondent’s disciplinary pol...,['Any personal records or notes related to the...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,10,Leonardo_WS.csv,10,Dismissal Details,Respondent dismissed appellant on 5 April 2024...,Predetermination Evidence During Pending Appea...,[{'text': 'The Respondent confirmed the dismis...,"['dismissal letter', 'detailed contextual expl...","[""Respondent's detailed reasons for dismissal""...",['Personal records of performance and ethical ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import json
import subprocess
from pathlib import Path
from pprint import pprint

import pandas as pd
from tqdm.notebook import tqdm

# =========================================================
# CONFIG
# =========================================================
Y_JSON_PATH = Path("/home/hello/Projects/Statements/output/Y_inferred.json")
MODEL = "mistral-small3.2:latest"
OLLAMA_CMD = ["ollama", "run", MODEL]

OUT_ENRICHED_JSON = Path("/home/hello/Projects/Statements/output/Y_inferred_enriched_with_tags.json")
OUT_ENRICHED_CSV  = Path("/home/hello/Projects/Statements/output/Y_inferred_x_tags.csv")

DEBUG = True   # prints per-Xi raw output if parsing fails

# Closed-set tags (your "needle list" for mapping)
ALLOWED_TAGS = [
    "upheld",
    "verbal_warning",
    "no_contemporaneous_evidence",
    "predetermination",
    "appeal_scope_limitation",
    "none",
]

TAG_DEFS = {
    "upheld": "appeal upheld / allowed / succeeds",
    "verbal_warning": "verbal warning mentioned as a disciplinary step or fact",
    "no_contemporaneous_evidence": "absence of notes/records or finding of no contemporaneous evidence",
    "predetermination": "decision appeared predetermined / outcome fixed / mind closed",
    "appeal_scope_limitation": "point not in grounds / outside scope / refused/declined to consider / limited to grounds",
    "none": "none of the above apply",
}

# =========================================================
# HELPERS
# =========================================================
def build_x_text(xid: str, xobj: dict) -> str:
    """
    Deterministic 'pretty' view of Xi test JSON -> text for LLM classification.
    Keep it stable so results are repeatable.
    """
    def join_list(xs):
        if not xs:
            return ""
        return "\n".join([f"- {str(x).strip()}" for x in xs if str(x).strip()])

    parts = []
    parts.append(f"X_ID: {xid}")
    parts.append(f"NAME: {xobj.get('name','')}")
    parts.append(f"SCOPE: {xobj.get('scope','')}")
    parts.append(f"DEFINITION: {xobj.get('definition','')}")
    parts.append(f"PATTERN: {xobj.get('pattern','')}")

    req = join_list(xobj.get("required_elements", []))
    if req:
        parts.append("REQUIRED_ELEMENTS:\n" + req)

    pos = join_list(xobj.get("positive_indicators", []))
    if pos:
        parts.append("POSITIVE_INDICATORS:\n" + pos)

    exc = join_list(xobj.get("excludes", []))
    if exc:
        parts.append("EXCLUDES:\n" + exc)

    return "\n\n".join([p for p in parts if p and p.strip()])


def make_prompt(x_text: str) -> str:
    tag_lines = "\n".join([f'- "{t}": {TAG_DEFS[t]}' for t in ALLOWED_TAGS])

    return f"""
TASK:
Given X_TEXT (a structured X-test definition), select all applicable TAGS from the allowed list.
Choose ONLY from the allowed tags. If none apply, select ONLY ["none"].

ALLOWED_TAGS (choose any):
{tag_lines}

RULES:
- For every selected tag (except "none"), provide:
  - confidence in [0,1]
  - evidence_quote copied verbatim from X_TEXT (max 200 chars)
  - negated: true if X_TEXT explicitly indicates the opposite (normally do not select if negated=true)
- If you cannot quote evidence from X_TEXT, do NOT select that tag.
- Return VALID JSON ONLY. No markdown, no commentary, no extra keys.

X_TEXT:
<<<
{x_text.strip()}
>>>

OUTPUT JSON SCHEMA:
{{
  "selected": [
    {{"tag": "...", "confidence": 0.0, "negated": false, "evidence_quote": "..."}}
  ]
}}
""".strip()


def run_tag_classifier(x_text: str) -> dict:
    prompt = make_prompt(x_text)

    proc = subprocess.run(
        OLLAMA_CMD,
        input=prompt,
        text=True,
        capture_output=True,
    )

    if proc.returncode != 0:
        raise RuntimeError(f"ollama returned {proc.returncode}: {proc.stderr[:400]}")

    raw = proc.stdout.strip()

    # Recover JSON if model adds wrapper text
    start = raw.find("{")
    end = raw.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object detected in model output.")

    json_str = raw[start : end + 1]
    out = json.loads(json_str)

    # Closed-set enforcement
    selected = out.get("selected", [])
    if not isinstance(selected, list):
        raise ValueError("Invalid output: 'selected' must be a list.")

    for item in selected:
        tag = item.get("tag")
        if tag not in ALLOWED_TAGS:
            raise ValueError(f"Invalid tag returned: {tag}")

    tags = [it.get("tag") for it in selected]
    if "none" in tags and len(tags) > 1:
        raise ValueError(f'Invalid output: "none" must be the only tag if present. Got: {tags}')

    return out


# =========================================================
# LOAD + RUN
# =========================================================
if not Y_JSON_PATH.exists():
    raise FileNotFoundError(f"Missing: {Y_JSON_PATH}")

y = json.loads(Y_JSON_PATH.read_text(encoding="utf-8"))

x_tests = y.get("x_tests", {})
if not isinstance(x_tests, dict) or not x_tests:
    raise ValueError("No 'x_tests' dict found in Y_inferred.json")

rows = []
enriched = {}

print(f"Loaded x_tests: {len(x_tests)} from {Y_JSON_PATH}")

for xid, xobj in tqdm(x_tests.items(), desc="Classifying X -> tags"):
    x_text = build_x_text(xid, xobj)

    try:
        cls = run_tag_classifier(x_text)
    except Exception as e:
        if DEBUG:
            print(f"\n[X={xid}] FAILED: {e}")
            print("----- X_TEXT (head) -----")
            print(x_text[:800])
        cls = {"selected": [{"tag": "none", "confidence": 0.0, "negated": False, "evidence_quote": ""}], "error": str(e)}

    # Persist per Xi
    enriched[xid] = cls

    # Flatten for DF
    selected = cls.get("selected", [])
    tags = [d.get("tag") for d in selected if d.get("tag")]
    rows.append({
        "X_id": xid,
        "X_name": xobj.get("name", ""),
        "tags_selected": "; ".join(tags),
        "n_tags": len([t for t in tags if t and t != "none"]),
        "raw_selected": json.dumps(selected, ensure_ascii=False),
    })

df_tags = pd.DataFrame(rows).sort_values(["n_tags", "X_id"], ascending=[False, True]).reset_index(drop=True)

# =========================================================
# WRITE BACK (enriched JSON + CSV)
# =========================================================
y_out = dict(y)
y_out["x_to_needle_tags"] = enriched

OUT_ENRICHED_JSON.write_text(json.dumps(y_out, indent=2, ensure_ascii=False), encoding="utf-8")
df_tags.to_csv(OUT_ENRICHED_CSV, index=False)

print("\nWrote:")
print(f" - {OUT_ENRICHED_JSON}")
print(f" - {OUT_ENRICHED_CSV}")

display(df_tags)

In [3]:
text = """
Definition of “Projects”, Management Instruction, and Environment Switching: Management Awareness and Reasonable Reliance
(January 2023 – March 2024)
04 January 2023 – Role definition and queue expectations
On 04 January 2023, internal management discussions disclosed under DSAR (page 176) expressly defined my role and priorities for the year ahead. Critically, those discussions make clear that queue work was not intended to be my primary or exclusive function. Management recorded that I was “needed on queue to help sometimes.” That wording is explicit and deliberate. Queue coverage was defined as occasional and situational, not as a standing priority. Management further confirmed that project work would, where necessary, take precedence over routine queue activity, stating that “projects and sometimes will use them over doing 1 and 2,” and concluding that this allocation reflected “what the team needs right now.” Management also discussed how this approach would be reflected formally, noting: “so in the review we stress on point 1 and say if you do that then you can do other projects.”
31 January 2023 – Contemporaneous confirmation of project-led role
That role definition was subsequently reflected in the Respondent’s own contemporaneous records dated 31 January 2023 (DSAR page 17). In internal communications, management described my role as involving direct client engagement on complex matters, noting that I had been “testing and improving client workflow throughout the year” and “taking ownership of some of our more difficult client projects.” The same records document that I was deliberately placed in front of clients to remediate significant regulatory reporting issues, including large-scale back-reporting exercises, retrieval and validation of historical files, testing of correction uploads, and guiding clients through submission processes until remediation was accepted. This work is explicitly described as client-facing and escalation-driven, rather than optional or ancillary.
Those records further confirm that this escalation work was undertaken because of its complexity and regulatory sensitivity, and that I worked alongside account management to resolve client issues under external regulatory pressure. The Respondent’s own wording therefore treats “projects” as structured escalation work delivered on behalf of clients, not as self-initiated side activities.
Progression through 2023 – Increased scope and environment switching
As I progressed through 2023 in a Senior Representative role, the volume and complexity of this management-directed project and client-escalation work increased. Delivering that work required operating across multiple Bloomberg environments, including switching between testing and live environments in order to validate workflows, test outputs, and support client and internal assignments. The frequency of environment switching increased directly in line with those responsibilities.
At that stage, I had no knowledge that working across environments could be associated with any adverse system effects. Nor was I informed by management, or otherwise, that this manner of working carried any operational risk or potential consequence. Environment switching was treated as a routine and accepted aspect of delivering the work I had been instructed to prioritise, and no concern, instruction, or guidance was raised in relation to it.
July 2023 – Identification of system effects without instruction
In or around July 2023, my Team Leader approached my desk to flag that certain adverse system effects were being observed. Until that point, I had no prior knowledge that such effects were occurring or that they were connected to my work. I immediately investigated the issue and, approximately one hour later that same day, explained my findings. I confirmed that the observed system behaviour arose as a consequence of the project and escalation work I had been instructed to prioritise and deliver, and that delivering that work required switching between different Bloomberg environments for testing, validation, and implementation purposes.
That discussion was limited to me openly and transparently explaining why the system behaviour had occurred. My Team Leader accepted that explanation and did not raise any issue with my working practices. No instruction was given to alter my working practices, restrict environment switching, reprioritise queue activity, or curtail the project work I had been assigned. No indication was given that continuing to work in the same manner could give rise to any performance or disciplinary consequences.
Mid-2023 interim evaluation to end-2023 – Workload intensity
Within weeks of that discussion, my 2023 interim performance evaluation took place. In that evaluation, management expressly reinforced that I should continue working on my projects, act as a resource for the team, engage stakeholders around that work, and assist colleagues. No criticism was raised, no concern was recorded, and no corrective instruction or guidance was issued.
From the mid-2023 interim evaluation through to the end of 2023, the volume and intensity of the project and escalation work assigned to me became excessive. Those priorities were repeatedly reinforced by my Team Leader during one-to-one meetings, with the clear focus being on delivery of projects and stakeholder support. As a result, I worked extended hours, including weekends, in order to meet those expectations and manage the regulatory and client pressure associated with that work.
Consistently with that role definition, I sent my Team Leader a detailed email setting out the status of the projects I was responsible for and their progress. That communication addressed only project and escalation work. It made no reference to queue activity, because queue coverage was not being treated as my primary responsibility at that time. The email reflects how my role and priorities were understood and applied in practice by both myself and management.
July 2023 – March 2024 – Reasonable reliance
Taken together, the absence of any instruction, warning, or restriction following July 2023, combined with the subsequent and repeated management endorsement to continue the same activities, meant that no reasonable employee in my position could have understood that their conduct required correction, restriction, or escalation.
Accordingly, from July 2023 through March 2024, I continued to perform my assigned work in the same manner, using the standard tools, permissions, and processes provided to me. I did so because the work remained required and ongoing, and because management—having full knowledge of both the nature of the work and the manner in which it was being delivered—had neither directed nor suggested that my approach was inappropriate or carried any adverse consequence.

"""

In [4]:
import json
import subprocess
from pprint import pprint

import pandas as pd

# =========================================================
# CONFIG
# =========================================================
MODEL = "mistral-small3.2:latest"
OLLAMA_CMD = ["ollama", "run", MODEL]

DEBUG = True  # print raw output if JSON parsing fails

ALLOWED_TAGS = [
    "upheld",
    "verbal_warning",
    "no_contemporaneous_evidence",
    "predetermination",
    "appeal_scope_limitation",
    "none",
]

TAG_DEFS = {
    "upheld": "appeal upheld / allowed / succeeds",
    "verbal_warning": "verbal warning mentioned as a disciplinary step or fact",
    "no_contemporaneous_evidence": "absence of notes/records or finding of no contemporaneous evidence",
    "predetermination": "decision appeared predetermined / outcome fixed / mind closed",
    "appeal_scope_limitation": "point not in grounds / outside scope / refused/declined to consider / limited to grounds",
    "none": "none of the above apply",
}

# =========================================================
# INPUT TEXT (PASTE HERE)
# =========================================================
TEXT = text.strip()
# =========================================================
# PROMPT
# =========================================================
tag_lines = "\n".join([f'- "{t}": {TAG_DEFS[t]}' for t in ALLOWED_TAGS])

PROMPT = f"""
TASK:
Given TEXT (from an ET/EAT decision or related document), select all applicable TAGS from the allowed list.
Choose ONLY from the allowed tags. If none apply, select ONLY ["none"].

ALLOWED_TAGS (choose any):
{tag_lines}

RULES:
- For every selected tag (except "none"), provide:
  - confidence in [0,1]
  - evidence_quote copied verbatim from TEXT (max 200 chars)
  - negated: true if TEXT explicitly indicates the opposite (normally do not select if negated=true)
- If you cannot quote evidence from TEXT, do NOT select that tag.
- Return VALID JSON ONLY. No markdown, no commentary, no extra keys.

TEXT:
<<<
{TEXT}
>>>

OUTPUT JSON SCHEMA:
{{
  "selected": [
    {{"tag": "...", "confidence": 0.0, "negated": false, "evidence_quote": "..."}}
  ]
}}
""".strip()

# =========================================================
# RUN
# =========================================================
proc = subprocess.run(
    OLLAMA_CMD,
    input=PROMPT,
    text=True,
    capture_output=True,
)

print("----- STDERR -----")
print(proc.stderr[:2000])

if proc.returncode != 0:
    raise RuntimeError(f"ollama returned {proc.returncode}")

raw = proc.stdout.strip()

# =========================================================
# PARSE JSON (recover if wrapped)
# =========================================================
start = raw.find("{")
end = raw.rfind("}")
if start == -1 or end == -1 or end <= start:
    if DEBUG:
        print("----- RAW STDOUT (head) -----")
        print(raw[:2000])
    raise ValueError("No JSON object detected in model output.")

json_str = raw[start : end + 1]

try:
    out = json.loads(json_str)
except json.JSONDecodeError:
    if DEBUG:
        print("----- JSON STR (head) -----")
        print(json_str[:1200])
        print("----- JSON STR (tail) -----")
        print(json_str[-1200:])
    raise

# =========================================================
# VALIDATE CLOSED SET
# =========================================================
selected = out.get("selected", [])
if not isinstance(selected, list):
    raise ValueError("Invalid output: 'selected' must be a list.")

for item in selected:
    tag = item.get("tag")
    if tag not in ALLOWED_TAGS:
        raise ValueError(f"Invalid tag returned: {tag}")

tags = [it.get("tag") for it in selected]
if "none" in tags and len(tags) > 1:
    raise ValueError(f'Invalid output: "none" must be the only tag if present. Got: {tags}')

print("\n----- PARSED OUTPUT -----")
pprint(out)

# =========================================================
# FLATTEN INTO A 1-ROW DF (handy for pipeline)
# =========================================================
flat = {
    "tags_selected": "; ".join(tags),
    "n_tags": len([t for t in tags if t and t != "none"]),
    "raw_selected": json.dumps(selected, ensure_ascii=False),
}
df_one = pd.DataFrame([flat])

display(df_one)

----- STDERR -----
⠙ ⠹ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ [?

----- PARSED OUTPUT -----
{'selected': [{'confidence': 0.8,
               'evidence_quote': 'On 04 January 2023, internal management '
                                 'discussions disclosed under DSAR (page 176) '
                                 'expressly defined my role and priorities for '
                                 'the year ahead.',
               'negated': False,
               'tag': 'no_contemporaneous_evidence'}]}


,tags_selected,n_tags,raw_selected
0,no_contemporaneous_evidence,1,"[{""tag"": ""no_contemporaneous_evidence"", ""confi..."
